In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:

# WARNING: This code will delete tables in your Lakehouse. This operation is irreversible!
# Please specify the table names you wish to drop in the 'tables_to_drop' list.

tables_to_drop = ["gold_dimcustomer", "gold_dimdate", "gold_dimproduct", "gold_factorders", "silver_orders"]  # Replace with your actual table names

for table_name in tables_to_drop:
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    print(f"Table {table_name} dropped (if it existed).")

In [ ]:
df = spark.read.format("csv").option("header","false").load("Files/Bronze/2025.csv")
# df now is a Spark DataFrame containing CSV data from "Files/Bronze/2019.csv".
display(df)

In [23]:
# Get the count of rows in the Spark DataFrame 'df'
row_count = df.count()
print(f"Number of rows in the DataFrame: {row_count}")

StatementMeta(, 5d51b447-bccb-49a5-b00d-bd73aa183037, 26, Finished, Available, Finished)

Number of rows in the DataFrame: 26804


## Transform Data for the Silver Layer

In [ ]:
from pyspark.sql.types import *
   
# Create the schema for the table
orderSchema = StructType([
     StructField("SalesOrderNumber", StringType()),
     StructField("SalesOrderLineNumber", IntegerType()),
     StructField("OrderDate", DateType()),
     StructField("CustomerName", StringType()),
     StructField("Email", StringType()),
     StructField("Item", StringType()),
     StructField("Quantity", IntegerType()),
     StructField("UnitPrice", FloatType()),
     StructField("Tax", FloatType())
     ])
   
# Import all files from the Files/Bronze folder of lakehouse
df = spark.read.format("csv").option("header", "false").schema(orderSchema).load("Files/Bronze/*.csv")
   
# Display the first 10 rows of the dataframe to preview your data
display(df.head(10))


In [26]:
from pyspark.sql.functions import when, lit, col, current_timestamp, input_file_name
 # Add columns IsFlagged, CreatedTS and ModifiedTS
df = df.withColumn("FileName", input_file_name()) \
     .withColumn("IsFlagged", when(col("OrderDate") < '2019-08-01',True).otherwise(False)) \
     .withColumn("CreatedTS", current_timestamp()).withColumn("ModifiedTS", current_timestamp())
   
 # Update CustomerName to "Unknown" if CustomerName is null or empty
df = df.withColumn("CustomerName", when((col("CustomerName").isNull() | (col("CustomerName")=="")),lit("Unknown")).otherwise(col("CustomerName")))


StatementMeta(, 5d51b447-bccb-49a5-b00d-bd73aa183037, 29, Finished, Available, Finished)

In [ ]:
# Define the schema for the orders table 
from pyspark.sql.types import *
from delta.tables import *
DeltaTable.createIfNotExists(spark) \
     .tableName("silver_orders") \
     .addColumn("SalesOrderNumber", StringType()) \
     .addColumn("SalesOrderLineNumber", IntegerType()) \
     .addColumn("OrderDate", DateType()) \
     .addColumn("CustomerName", StringType()) \
     .addColumn("Email", StringType()) \
     .addColumn("Item", StringType()) \
     .addColumn("Quantity", IntegerType()) \
     .addColumn("UnitPrice", FloatType()) \
     .addColumn("Tax", FloatType()) \
     .addColumn("FileName", StringType()) \
     .addColumn("IsFlagged", BooleanType()) \
     .addColumn("CreatedTS", DateType()) \
     .addColumn("ModifiedTS", DateType()) \
     .execute()


In [28]:
# Update existing records and insert new ones based on a condition defined by the columns SalesOrderNumber, OrderDate, CustomerName, and Item.
from delta.tables import *
deltaTable = DeltaTable.forPath(spark, 'Tables/silver_orders')
dfUpdates = df
deltaTable.alias('silver') \
  .merge(
    dfUpdates.alias('updates'),
    'silver.SalesOrderNumber = updates.SalesOrderNumber and silver.OrderDate = updates.OrderDate and silver.CustomerName = updates.CustomerName and silver.Item = updates.Item'
  ) \
   .whenMatchedUpdate(set =
    {
         
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "SalesOrderNumber": "updates.SalesOrderNumber",
      "SalesOrderLineNumber": "updates.SalesOrderLineNumber",
      "OrderDate": "updates.OrderDate",
      "CustomerName": "updates.CustomerName",
      "Email": "updates.Email",
      "Item": "updates.Item",
      "Quantity": "updates.Quantity",
      "UnitPrice": "updates.UnitPrice",
      "Tax": "updates.Tax",
      "FileName": "updates.FileName",
      "IsFlagged": "updates.IsFlagged",
      "CreatedTS": "updates.CreatedTS",
      "ModifiedTS": "updates.ModifiedTS"
    }
  ) \
  .execute()


StatementMeta(, 5d51b447-bccb-49a5-b00d-bd73aa183037, 31, Finished, Available, Finished)

In [ ]:
%%sql
SELECT *
FROM silver_orders
LIMIT 10;


## Gold Layer dimCustomer

In [30]:
from pyspark.sql.types import *
from delta.tables import * 
# Create customer dimension delta table
DeltaTable.createIfNotExists(spark) \
     .tableName("gold_dimCustomer") \
     .addColumn("CustomerName", StringType()) \
     .addColumn("Email",  StringType()) \
     .addColumn("First", StringType()) \
     .addColumn("Last", StringType()) \
     .addColumn("CustomerID", LongType()) \
     .execute()


StatementMeta(, 5d51b447-bccb-49a5-b00d-bd73aa183037, 33, Finished, Available, Finished)

In [31]:
from pyspark.sql.functions import col, split
   
 # Create customer dataframe
dfdimCustomer = df.dropDuplicates(["CustomerName","Email"]).select(col("CustomerName"),col("Email")) \
     .withColumn("First",split(col("CustomerName"), " ").getItem(0)) \
     .withColumn("Last",split(col("CustomerName"), " ").getItem(1))
 # Display the first 10 rows of the dataframe to preview your data
display(dfdimCustomer.head(10))


StatementMeta(, 5d51b447-bccb-49a5-b00d-bd73aa183037, 34, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 0a39978a-2d58-41af-a40b-c76559e6009f)

In [32]:
from pyspark.sql.functions import monotonically_increasing_id, col, when, coalesce, max, lit
dfdimCustomer_temp = spark.read.table("gold_dimCustomer")
MAXCustomerID = dfdimCustomer_temp.select(coalesce(max(col("CustomerID")),lit(0)).alias("MAXCustomerID")).first()[0]
dfdimCustomer = dfdimCustomer.join(dfdimCustomer_temp,(dfdimCustomer.CustomerName == dfdimCustomer_temp.CustomerName) & (dfdimCustomer.Email == dfdimCustomer_temp.Email), "left_anti")
dfdimCustomer = dfdimCustomer.withColumn("CustomerID",monotonically_increasing_id() + MAXCustomerID + 1)
# Display the first 10 rows of the dataframe to preview your data
display(dfdimCustomer.head(10))


StatementMeta(, 5d51b447-bccb-49a5-b00d-bd73aa183037, 35, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b577049d-b3e5-483b-8350-0ca75f19a66e)

In [33]:
from delta.tables import *
deltaTable = DeltaTable.forPath(spark, 'Tables/gold_dimcustomer') 
dfUpdates = dfdimCustomer 
deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.CustomerName = updates.CustomerName AND gold.Email = updates.Email'
  ) \
   .whenMatchedUpdate(set =
    {
         
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "CustomerName": "updates.CustomerName",
      "Email": "updates.Email",
      "First": "updates.First",
      "Last": "updates.Last",
      "CustomerID": "updates.CustomerID"
    }
  ) \
  .execute()


StatementMeta(, 5d51b447-bccb-49a5-b00d-bd73aa183037, 36, Finished, Available, Finished)

## Gold Layer: dimDate Dimensin 

In [8]:
spark.sql("DROP TABLE IF EXISTS gold_dimdate")

StatementMeta(, c51fc448-0632-44a8-a930-a34688e0425e, 10, Finished, Available, Finished)

DataFrame[]

In [ ]:
from pyspark.sql.types import *
from delta.tables import* 
 # Define the schema for the dimdate_gold table
DeltaTable.createIfNotExists(spark) \
     .tableName("gold_dimdate") \
     .addColumn("OrderDate", DateType()) \
     .addColumn("Day", IntegerType()) \
     .addColumn("Month", IntegerType()) \
     .addColumn("Year", IntegerType()) \
     .addColumn("mmmyyyy", StringType()) \
     .addColumn("yyyymm", StringType()) \
     .execute()


In [ ]:
from pyspark.sql.functions import explode, sequence, to_date, year, month, dayofmonth, date_format, col, lit, expr
from pyspark.sql.types import DateType, IntegerType, StringType

start_date = "2019-01-01"
end_date   = "2025-12-31"

dfDimDate = (
    spark
    .range(1)
    .select(
        explode(
            sequence(
                to_date(lit(start_date)),
                to_date(lit(end_date)),
                expr("interval 1 day")
            )
        ).alias("OrderDate")
    )
    .select(
        col("OrderDate").cast(DateType()).alias("OrderDate"),
        dayofmonth("OrderDate").cast(IntegerType()).alias("Day"),
        month("OrderDate").cast(IntegerType()).alias("Month"),
        year("OrderDate").cast(IntegerType()).alias("Year"),
        date_format(col("OrderDate"), "MMM-yyyy").cast(StringType()).alias("mmmyyyy"),
        date_format(col("OrderDate"), "yyyyMM").cast(StringType()).alias("yyyymm")
    )
    .dropDuplicates(["OrderDate"])   # guardrail
    .select("OrderDate","Day","Month","Year","mmmyyyy","yyyymm")  # enforce column order
    .orderBy("OrderDate")
)



## Validate No Gaps in dimDate

In [ ]:
from pyspark.sql.functions import lag, datediff
from pyspark.sql.window import Window

w = Window.orderBy(col("OrderDate"))
gaps = (
    dfDimDate
    .select(
        col("OrderDate"),
        lag("OrderDate").over(w).alias("PrevDate")
    )
    .where(col("PrevDate").isNotNull())
    .where(datediff(col("OrderDate"), col("PrevDate")) != 1)
)

display(gaps)


In [ ]:
dfDimDate.write \
      .format("delta")  \
      .mode("overwrite") \
      .option("overwriteSchema", "false") \
      .saveAsTable("gold_dimdate")


In [ ]:
dfDimDate.show()

from pyspark.sql.functions import col, dayofmonth, month, year, date_format
 # Create dataframe for dimDate_gold
dfdimDate_gold = df.dropDuplicates(["OrderDate"]).select(col("OrderDate"), \
         dayofmonth("OrderDate").alias("Day"), \
         month("OrderDate").alias("Month"), \
         year("OrderDate").alias("Year"), \
         date_format(col("OrderDate"), "MMM-yyyy").alias("mmmyyyy"), \
         date_format(col("OrderDate"), "yyyyMM").alias("yyyymm"), \
     ).orderBy("OrderDate")
 # Display the first 10 rows of the dataframe to preview your data
display(dfdimDate_gold.head(10))

from delta.tables import *  
deltaTable = DeltaTable.forPath(spark, 'Tables/gold_dimdate')
dfUpdates = dfdimDate_gold
deltaTable.alias('silver') \
   .merge(
     dfUpdates.alias('updates'),
     'silver.OrderDate = updates.OrderDate'
   ) \
    .whenMatchedUpdate(set =
     {
         
     }
   ) \
  .whenNotMatchedInsert(values =
     {
       "OrderDate": "updates.OrderDate",
       "Day": "updates.Day",
       "Month": "updates.Month",
       "Year": "updates.Year",
       "mmmyyyy": "updates.mmmyyyy",
       "yyyymm": "yyyymm"
     }
   ) \
   .execute()


In [ ]:
from pyspark.sql.types import *
from delta.tables import * 
DeltaTable.createIfNotExists(spark) \
    .tableName("gold_dimproduct") \
    .addColumn("ItemName", StringType()) \
    .addColumn("ItemID", LongType()) \
    .addColumn("ItemDesc", StringType()) \
    .execute()


In [ ]:
from pyspark.sql.functions import col, split, lit   
# Create product dataframe  
dfdimProduct_silver = df.dropDuplicates(["Item"]).select(col("Item")) \
    .withColumn("ItemName",split(col("Item"), ", ").getItem(0)) \
    .withColumn("ItemDesc",when((split(col("Item"), ", ").getItem(1).isNull() | (split(col("Item"), ", ").getItem(1)=="")),lit("")).otherwise(split(col("Item"), ", ").getItem(1)))
# Display the first 10 rows of the dataframe to preview your data
display(dfdimProduct_silver.head(10))


In [ ]:
from pyspark.sql.functions import monotonically_increasing_id, col, lit, max, coalesce
#dfdimProduct_temp = dfdimProduct_silver
dfdimProduct_temp = spark.read.table("gold_dimProduct") 
MAXProductID = dfdimProduct_temp.select(coalesce(max(col("ItemID")),lit(0)).alias("MAXItemID")).first()[0]
dfdimProduct_gold = dfdimProduct_silver.join(dfdimProduct_temp,(dfdimProduct_silver.ItemName == dfdimProduct_temp.ItemName) & (dfdimProduct_silver.ItemDesc == dfdimProduct_temp.ItemDesc), "left_anti")
   
dfdimProduct_gold = dfdimProduct_gold.withColumn("ItemID",monotonically_increasing_id() + MAXProductID + 1)   
# Display the first 10 rows of the dataframe to preview your data
display(dfdimProduct_gold.head(10))


In [ ]:
 #to wrap the Product dimension up, we need to insert records from the dataframe into the delta table:
from delta.tables import *
deltaTable = DeltaTable.forPath(spark, 'Tables/gold_dimproduct')        
dfUpdates = dfdimProduct_gold       
deltaTable.alias('silver') \
  .merge(
        dfUpdates.alias('updates'),
        'silver.ItemName = updates.ItemName AND silver.ItemDesc = updates.ItemDesc'
        ) \
        .whenMatchedUpdate(set =
        {
               
        }
        ) \
        .whenNotMatchedInsert(values =
         {
          "ItemName": "updates.ItemName",
          "ItemDesc": "updates.ItemDesc",
          "ItemID": "updates.ItemID"
          }
          ) \
          .execute()


## Create the Fact table

In [ ]:
# Now that we have all our dimension tables in place, the only remaining step is to create a fact table. 
# We will start by defining the table structure:
from pyspark.sql.types import *
from delta.tables import * 
DeltaTable.createIfNotExists(spark) \
    .tableName("gold_factorders") \
    .addColumn("CustomerID", LongType()) \
    .addColumn("ItemID", LongType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", FloatType()) \
    .addColumn("Tax", FloatType()) \
    .execute()


In [ ]:
from pyspark.sql.functions import col  
dfdimCustomer_temp = spark.read.table("gold_dimCustomer")
dfdimProduct_temp = spark.read.table("gold_dimProduct") 
df = df.withColumn("ItemName",split(col("Item"), ", ").getItem(0)) \
    .withColumn("ItemDesc",when((split(col("Item"), ", ").getItem(1).isNull() | (split(col("Item"), ", ").getItem(1)=="")),lit("")).otherwise(split(col("Item"), ", ").getItem(1))) \
# Create Sales_gold dataframe
dffactSales_gold = df.alias("df1").join(dfdimCustomer_temp.alias("df2"),(df.CustomerName == dfdimCustomer_temp.CustomerName) & (df.Email == dfdimCustomer_temp.Email), "left") \
        .join(dfdimProduct_temp.alias("df3"),(df.ItemName == dfdimProduct_temp.ItemName) & (df.ItemDesc == dfdimProduct_temp.ItemDesc), "left") \
    .select(col("df2.CustomerID") \
        , col("df3.ItemID") \
        , col("df1.OrderDate") \
        , col("df1.Quantity") \
        , col("df1.UnitPrice") \
        , col("df1.Tax") \
    ).orderBy(col("df1.OrderDate"), col("df2.CustomerID"), col("df3.ItemID"))
# Display the first 10 rows of the dataframe to preview your data
display(dffactSales_gold.head(10))


In [ ]:
from delta.tables import * 
deltaTable = DeltaTable.forPath(spark, 'Tables/gold_factorders')
dfUpdates = dffactSales_gold
deltaTable.alias('silver') \
  .merge(
    dfUpdates.alias('updates'),
    'silver.OrderDate = updates.OrderDate AND silver.CustomerID = updates.CustomerID AND silver.ItemID = updates.ItemID'
  ) \
   .whenMatchedUpdate(set =
    {
         
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "CustomerID": "updates.CustomerID",
      "ItemID": "updates.ItemID",
      "OrderDate": "updates.OrderDate",
      "Quantity": "updates.Quantity",
      "UnitPrice": "updates.UnitPrice",
      "Tax": "updates.Tax"
    }
  ) \
  .execute()


In [ ]:
%%sql
SELECT * FROM gold_factorders
WHERE OrderDate BETWEEN DATE('2026-01-01') and DATE('2026-12-31')
LIMIT 10


In [ ]:
%%sql
UPDATE gold_factorders
SET CustomerID = 9999
WHERE CUSTOMERID = 54

In [ ]:
%%sql
SELECT * FROM gold_factorders
WHERE CustomerID = 9999